# UED baselines on JaxUED mazes - RunPod A100

Reproduces the three reference curricula from `docs/TASK.md` - **DR**, **PLR-perp** and **ACCEL** -
with the student frozen, then leaves the machinery in place for new teacher ideas.

Order of business, and why:

| # | Step | Why it comes here |
|---|------|-------------------|
| 1 | Probe + bootstrap | pinned deps in an isolated venv on the persistent volume |
| 2 | Smoke runs | catch plumbing errors in ~2 min, and *measure* the cost of a full run |
| 3 | Parity vs upstream | prove our refactor reproduces `maze_dr.py` / `maze_plr.py` bit-for-bit |
| 4 | Checkpoint compatibility | prove the graders' evaluation harness can load what we submit |
| 5 | Concurrency probe | the model is tiny; one A100 may fit several runs at once |
| 6 | Full sweep | 30000 updates per run, detached and resumable |
| 7 | Results | curves, per-level breakdown, the table for the report |

Nothing here trains in the kernel. Every run is a detached subprocess writing to
`runs/<run_name>/<seed>/`, so restarting this notebook - or losing the browser tab - does not
kill a two-hour job.

## 1. Probe and bootstrap

In [ ]:
import json, os, pathlib, subprocess, sys, time

REPO = pathlib.Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

# RunPod's persistent volume, if we are on one. Everything heavy lives here so a
# pod restart costs nothing: the venv, the checkpoints and the metrics.
WORKSPACE = pathlib.Path("/workspace") if pathlib.Path("/workspace").exists() else REPO
OUT_DIR = WORKSPACE / "tlab_ued"
PY = WORKSPACE / "venvs" / "jaxued" / "bin" / "python"   # created by bootstrap.sh
JAXUED = REPO / "third_party" / "jaxued"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def child_env(**overrides):
    """Environment for a subprocess running the *venv* interpreter.

    The kernel is a different Python installation and some of its variables are
    hostile to the child. `MPLBACKEND` is the sharp one: Jupyter sets it to
    `module://matplotlib_inline.backend_inline`, which the venv does not have,
    and gymnax imports `matplotlib.pyplot` at import time - so inheriting it
    turns every `import jaxued.environments` into `ValueError: Key backend`.
    `PYTHONPATH`/`PYTHONHOME` would splice the kernel's site-packages (numpy 2,
    another jax) into the venv.
    """
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME")}
    env["MPLBACKEND"] = "Agg"
    env.setdefault("WANDB_MODE", "offline")
    env.setdefault("PYTHONUNBUFFERED", "1")
    env.update(overrides)
    return env


def sh(cmd, cwd=None, env=None, check=True):
    """Run a command, streaming its output into the notebook."""
    printable = cmd if isinstance(cmd, str) else " ".join(str(c) for c in cmd)
    print("$", printable, flush=True)
    proc = subprocess.Popen(
        cmd if isinstance(cmd, str) else [str(c) for c in cmd],
        shell=isinstance(cmd, str), cwd=cwd, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env=child_env(**(env or {})),
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode:
        raise RuntimeError(f"exit {proc.returncode}: {printable}")
    return proc.returncode


print("repo      :", REPO)
print("workspace :", WORKSPACE)
print("outputs   :", OUT_DIR)
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)

In [ ]:
# Idempotent: clones jaxued at the pinned SHA, builds the venv, installs the
# exact versions from requirements.txt. First run ~5 min (CUDA wheels), then instant.
sh(["bash", "scripts/bootstrap.sh", WORKSPACE])
assert PY.exists(), f"bootstrap did not produce {PY}"

In [ ]:
# Sanity: the pinned stack sees the A100, and our package imports.
sh([PY, "-c", "import jax, jaxued, tlab_ued; print(jax.__version__, jax.devices())"])
sh([PY, "-c", "from tlab_ued.teachers import TEACHERS; from tlab_ued.scoring import SCORE_FUNCTIONS;"
              " print('teachers:', sorted(TEACHERS)); print('scores:', sorted(SCORE_FUNCTIONS))"])

## 2. Smoke runs

500 updates per teacher (~2 min each). This is a plumbing and timing check only - `docs/TASK.md`
warns that the *ordering* of methods on short runs is not the ordering at full budget, so nothing
here says anything about which curriculum is better.

What it does tell us is `time_delta` per eval step, which extrapolates directly to the cost of the
full 30000-update runs (120 eval steps).

Smoke runs are named `<method>_smoke` and write to their own `runs/` and `checkpoints/`
directories, so they can never be confused with - or block - the real sweep.

In [ ]:
for preset in ("dr", "plr", "accel"):
    sh([PY, "-m", "tlab_ued.train", "--preset", preset, "--smoke", "--out_dir", OUT_DIR])

In [ ]:
from tlab_ued.analysis import load_runs, throughput

smoke = load_runs(OUT_DIR)
display(throughput(smoke))
print("\nRule of thumb: hours_per_30k_updates x 3 methods x n_seeds = GPU-hours for the sweep.")

## 3. Parity against upstream

Our trainer is a refactor of `examples/maze_plr.py` and `examples/maze_dr.py` with the teacher
lifted out into a plugin. The refactor is only safe if it changes nothing: same frozen student,
same `jax.random.split` pattern, same order of operations.

This runs both implementations in-process on the same config and seed, captures what each passes
to `wandb.log`, and diffs the scalars. **Expected: every difference exactly 0.0.** A non-zero diff
means a seam was cut wrong, and it is much cheaper to find out now than after nine hours of GPU
time.

In [ ]:
sh([PY, "-m", "tlab_ued.parity", "--presets", "dr", "plr", "accel",
    "--num_updates", "500", "--out_dir", OUT_DIR / "_parity"])

## 4. Checkpoint compatibility

The assignment evaluates our submitted checkpoints with *its own* harness, which restores the
orbax checkpoint untargeted and reads `loaded["params"]` into the original `ActorCritic`. Our
checkpoints carry extra teacher state, so let us prove that does not matter by evaluating one of
them with the **upstream** script, unmodified.

If this cell prints solve rates, the submission will load.

In [ ]:
ckpt = OUT_DIR / "checkpoints" / "accel_maxmc_smoke" / "0"
print("evaluating", ckpt, "with upstream examples/maze_plr.py\n")
sh([PY, "examples/maze_plr.py", "--mode", "eval",
    "--checkpoint_directory", ckpt, "--eval_num_attempts", "2"], cwd=JAXUED)

## 5. Concurrency probe (optional)

The maze student is small: a single run does not saturate an A100. If two or three trainers share
the card, wall-clock time for the whole sweep can drop substantially even though each individual
run gets slower.

Each process is capped with `XLA_PYTHON_CLIENT_MEM_FRACTION` so the first one does not preallocate
the card. Skip this cell if you would rather not spend ~10 minutes measuring.

In [ ]:
from tlab_ued.sweep import Job, run_sweep
from tlab_ued.analysis import load_runs

def probe(n_parallel):
    jobs = [Job("accel", seed=900 + n_parallel * 10 + i) for i in range(n_parallel)]
    start = time.time()
    run_sweep(jobs, max_parallel=n_parallel, poll_seconds=10, python=str(PY),
              mem_fraction=round(0.85 / n_parallel, 2) if n_parallel > 1 else None,
              out_dir=str(OUT_DIR), smoke=True)
    return time.time() - start

wall = {n: probe(n) for n in (1, 2)}
for n, seconds in wall.items():
    print(f"{n} concurrent smoke run(s): {seconds:.0f}s wall -> {n / seconds * 500:.1f} updates/s aggregate")

MAX_PARALLEL = max(wall, key=lambda n: n / wall[n])
print("\nchosen MAX_PARALLEL =", MAX_PARALLEL)

## 6. Full sweep

30000 updates per run, `--checkpoint_save_interval 17` (the assignment's value: one checkpoint
every 17 eval steps = every 4250 updates, ~7 per run).

Start with **seed 0 for all three baselines**, now that the smoke timing says what that costs.
Adding seeds is editing `SEEDS` and re-running the cell - jobs that already finished are skipped,
and a job that was interrupted resumes from its last checkpoint.

In [ ]:
from tlab_ued.sweep import Job, sweep_step, job_status, format_status

PRESETS = ("dr", "plr", "accel")
SEEDS = (0,)                    # extend to (0, 1, 2) once seed 0 has landed
MAX_PARALLEL = globals().get("MAX_PARALLEL", 1)

JOBS = [Job(preset=p, seed=s) for s in SEEDS for p in PRESETS]
COMMON = dict(out_dir=str(OUT_DIR))

statuses = sweep_step(JOBS, max_parallel=MAX_PARALLEL, python=str(PY),
                      mem_fraction=round(0.85 / MAX_PARALLEL, 2) if MAX_PARALLEL > 1 else None,
                      **COMMON)
print(format_status(statuses))

In [ ]:
# Progress. Re-run freely: it reads the filesystem and also starts the next job
# whenever a slot frees up, so this cell doubles as the scheduler.
statuses = sweep_step(JOBS, max_parallel=MAX_PARALLEL, python=str(PY),
                      mem_fraction=round(0.85 / MAX_PARALLEL, 2) if MAX_PARALLEL > 1 else None,
                      **COMMON)
print(format_status(statuses), "\n")
for s in statuses:
    if s["state"] in ("running", "interrupted") and os.path.exists(s["log"]):
        tail = open(s["log"]).read().splitlines()[-3:]
        print(f"--- {s['run_name']} seed {s['seed']}")
        print("\n".join("    " + line for line in tail))

## 7. Results

Everything below reads the per-run `metrics.csv` files - no GPU and no wandb needed, so the same
cells work on a laptop after copying `runs/` off the pod.

Read these against the reference numbers in `docs/TASK.md`. At full budget the expected ordering
is ACCEL > PLR-perp > DR; if a short run says otherwise, that is the short run talking.

In [ ]:
import matplotlib.pyplot as plt
from tlab_ued.analysis import load_runs, final_table, per_level_table, plot_curves, plot_per_level

df = load_runs(OUT_DIR)
# keep the real runs: drop smoke/probe runs and parity runs
df = df[~df["run_name"].str.endswith("_smoke")]
df = df[~df["run_name"].str.startswith("parity")]

print("runs found:", df.groupby("run_name")["seed"].nunique().to_dict())
display(final_table(df, last_k=3))     # average the last 3 evals: less noise than a single point

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 10))
plot_curves(df, ax=axes[0])
plot_per_level(df, ax=axes[1], last_k=3)
fig.tight_layout()

figs = OUT_DIR / "results" / "figs"
figs.mkdir(parents=True, exist_ok=True)
fig.savefig(figs / "baselines.png", dpi=150, bbox_inches="tight")
print("saved", figs / "baselines.png")

In [ ]:
display(per_level_table(df, last_k=3).round(3))

### Package the checkpoints for submission

The assignment asks for `checkpoints/<run_name>/<seed>` for our method **and** our ACCEL run, per
seed. These were written throughout training by `--checkpoint_save_interval 17`.

In [ ]:
import shutil

for d in sorted((OUT_DIR / "checkpoints").glob("*/*")):
    size_mb = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e6
    steps = sorted(int(p.name) for p in (d / "models").glob("[0-9]*") if p.is_dir())
    print(f"{str(d.relative_to(OUT_DIR)):<40} {size_mb:7.1f} MB   steps={steps}")

archive = shutil.make_archive(str(OUT_DIR / "checkpoints_submission"), "gztar",
                              root_dir=OUT_DIR, base_dir="checkpoints")
print("\narchive:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")

## Next: a new teacher

The baselines are the control group. To try an idea:

1. **New score function** - `src/tlab_ued/scoring.py`: write a function taking `(config, signals)`,
   decorate it with `@register_score_fn("my_score")`, run with `--score_function my_score`.
   `RolloutSignals` carries the whole rollout (obs, actions, logits, values, advantages, returns,
   levels), so a score can use signals MaxMC and PVL ignore.
2. **New curriculum logic** - `src/tlab_ued/teachers/my_idea.py`: subclass `Teacher`, implement
   `wrap_env`, `init_teacher_state`, `branches` and `select_branch`, register it in
   `teachers/__init__.py`, run with `--teacher my_idea`.
3. **New level generation or mutation** - `src/tlab_ued/levels.py`, via `@register_generator` /
   `@register_mutator`.

Then add it to `PRESETS` in `config.py`, put it in `PRESETS` in the sweep cell above, and the same
parity/compat/sweep/plot machinery applies unchanged. The student stays frozen -
`assert_student_frozen` fails the run if it does not.